In [7]:
!pip install sqlalchemy
!pip install mysql-connector-python


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd

In [9]:
# Load CSV
df = pd.read_csv("../data/EDA_Analysis.csv")

In [10]:
df.columns

Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Response', 'Complain', 'Country', 'Age', 'Total_Spend',
       'Total_Purchases', 'Children', 'Age_Group', 'Family_Size',
       'Living_With', 'Customer_Tenure', 'Total_Accepted_Campaigns',
       'Customer_Segment', 'Purchase_Frequency', 'Web_Engagement', 'Is_Parent',
       'Income_Group', 'Engagement_Segment', 'Spending_Segment',
       'Response_Segment', 'Family_Segment', 'High_Value', 'Under_Served',
       'Deal_Seeker', 'Loyal_Customer', 'At_Risk', 'New_Customer', 'Year',
       'Month'],
      dtype='object')

In [ ]:
df.value_counts

In [11]:
df.shape

(56000, 54)

                ┌─────────────────────┐
                │    dim_customer   Master Table  │
                │---------------------│
                │ customer_id (PK)    │
                │ age, income         │
                │ education           │
                │ marital_status      │
                │ country             │
                └─────────┬───────────┘
                          │
        ┌─────────────────┼─────────────────┐
        │                 │                 │
┌──────────────┐ ┌────────────────┐ ┌──────────────────┐
│ fact_spending│ │ fact_campaign  │ │ dim_time (opt)   │
│--------------│ │----------------│ │------------------│
│ customer_id FK│ │ customer_id FK │ │ year, month      │
│ total_spend   │ │ response       │ │ date hierarchy   │
│ purchases     │ │ campaigns      │ └──────────────────┘
└──────────────┘ └────────────────┘

DATA are Two types:

Who=Customer
What=Actions

Idea:

1 Customer = many actions

dim_customer (center)
       ↓
fact tables (spending + campaign)


RELATIONSHIP:
dim_customer (1)
     |
     |  customer_id
     ↓
fact_spending (many rows)
fact_campaign (many rows)

1.Dim_customer (MASTER TABLE)

customer ki identity + profile

Data 
(Who is the Coustomer?)

Age
Income
Educations
Marital_status
Country


fact_spending (WHAT THEY BUY)

how many rupess Custmoer Spent
Data:
(what product customer buy and how many?)

Total Spend
Wine/Fruits/Meat spending
Purchases
Web/Store purchases



fact_campaign (HOW THEY REACTED)

Response of Marketing campaign

data
(what reaction of customer?)

Data:
Response (Yes/No)
Campaign 1–5 accepted
Complaint
At Risk / Loyal / New

dim_time 

Data Analysis

Data
(According to time Trend Analysis?)
Year
Month

SIMPLE ONE-LINE SUMMARY

👉 dim_customer = Who the customer is
👉 fact_spending = What they buy
👉 fact_campaign = How they react

In [16]:
import mysql.connector

# Load CSV
df = pd.read_csv("../data/EDA_Analysis.csv")

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="mysql",   # your password
    port=3305,           # your port
   database="Marketing_campaign_analysis_DB" # dataBase
                   
)

cursor = conn.cursor()

In [17]:
for _, row in df.iterrows():
    cursor.execute("""
    INSERT INTO dim_customer VALUES
    (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        row["ID"],
        row["Year_Birth"],
        row["Age"],
        row["Education"],
        row["Marital_Status"],
        row["Country"],
        row["Income"],
        row["Kidhome"],
        row["Teenhome"],
        row["Children"],
        row["Family_Size"],
        row["Living_With"],
        row["Is_Parent"],
        row["Customer_Tenure"]
    ))

conn.commit()

In [18]:
for _, row in df.iterrows():
    cursor.execute("""
    INSERT INTO fact_spending VALUES
    (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        row["ID"],
        row["MntWines"],
        row["MntFruits"],
        row["MntMeatProducts"],
        row["MntFishProducts"],
        row["MntSweetProducts"],
        row["MntGoldProds"],
        row["Total_Spend"],
        row["Total_Purchases"],
        row["Purchase_Frequency"],
        row["NumDealsPurchases"],
        row["NumWebPurchases"],
        row["NumCatalogPurchases"],
        row["NumStorePurchases"],
        row["NumWebVisitsMonth"]
    ))

conn.commit()

In [19]:
for _, row in df.iterrows():
    cursor.execute("""
    INSERT INTO fact_campaign VALUES
    (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        row["ID"],
        row["Recency"],
        row["Response"],
        row["Complain"],
        row["AcceptedCmp1"],
        row["AcceptedCmp2"],
        row["AcceptedCmp3"],
        row["AcceptedCmp4"],
        row["AcceptedCmp5"],
        row["Total_Accepted_Campaigns"],
        row["Customer_Segment"],
        row["Engagement_Segment"],
        row["Spending_Segment"],
        row["Response_Segment"],
        row["Family_Segment"],
        row["High_Value"],
        row["Under_Served"],
        row["Deal_Seeker"],
        row["Loyal_Customer"],
        row["At_Risk"],
        row["New_Customer"],
        row["Year"],
        row["Month"]
    ))

conn.commit()

CSV DATA
   ↓
MySQL DATABASE (marketing_db)
   ↓
dim_customer
fact_spending
fact_campaign
   ↓
Power BI / Streamlit Dashboard

CUSTOMER OVERVIEW INSIGHTS

Question

Q1: How many total customers are in the database?
 
 select count(*) from dim_customer;
 56000

Q2:What is the average income of customers?

select avg(income) from dim_customer;
57243.79527723231

Q3:What is the distribution of customers by education level?

SELECT education, COUNT(customer_id) AS total_customers
FROM dim_customer
GROUP BY education
ORDER BY total_customers DESC;

0	2n Cycle	 5966
1	Basic	     4182
2	Graduation   22741
3	Master	     10530
4	PhD  	     12581

Q4:What is the average age of customers by country?
 
 SELECT country,avg(age) As Total_customer
 From dim_customer
 GROUP BY country
 ORDER By Total_customer DESC;

0	Australia	53.7138
1	USA	        53.6391
2	Germany	    52.9551
3	Canada	    52.8496
4	India	    52.4605
5	Spain	    51.4815
6	Saudi Arabia51.4583
7	Mexico	    51.4046

SPENDING INSIGHTS

1.Who are the top 10 highest spending customers?

SELECT 
    d.customer_id AS Top_customer,
   f.total_spend
FROM dim_customer d
JOIN fact_spending f
ON d.customer_id = f.customer_id
ORDER BY total_spend DESC
LIMIT 10;



0	15231247	2395.5
1	5971890	2395.5
2	11926134	2395.5
3	1869612	2395.5
4	509614	2395.5
5	15178963	2395.5
6	6231050	2395.5
7	765261	2395.5
8	1745090	2395.5
9	2462308	2395.5


2. Which country has the highest total spending?

SELECT 
    d.country,
    SUM(f.total_spend) AS Total_spending
FROM dim_customer d
JOIN fact_spending f
ON d.customer_id = f.customer_id
GROUP BY d.country
ORDER BY Total_spending DESC;


0	Spain	         9259027
1	Canada	         6262792
2	Saudi Arabia	 5482349
3	Australia	     4669446
4	India	         3467467.5
5	Germany	         3117343
6	USA	             2858707
7	Mexico	         553047

Q3:What is the average spending by education level?

  select d.education,
        avg(f.total_spend)As Avg_Spending
 FROM dim_customer d
 JOIN fact_spending f
 ON d.customer_id=f.customer_id
 GROUP BY d.education
 ORDER by  Avg_Spending DESC;


0	Graduation	742.9869838617475
1	Master	    644.8897910731245
2	PhD	        572.761545187187
3	2n Cycle	509.84135098893734
4	Basic	    415.01566236250596



Q 4: Which product category contributes the most to total spend?

SELECT 'Wines' AS product_category,
       SUM(mnt_wines) AS total_spend
FROM fact_spending

UNION ALL

SELECT 'Fruits',
       SUM(mnt_fruits)
FROM fact_spending

UNION ALL

SELECT 'Meat Products',
       SUM(mnt_meat_products)
FROM fact_spending

UNION ALL

SELECT 'Fish Products',
       SUM(mnt_fish_products)
FROM fact_spending

UNION ALL

SELECT 'Sweet Products',
       SUM(mnt_sweet_products)
FROM fact_spending

UNION ALL

SELECT 'Gold Products',
       SUM(mnt_gold_prods)
FROM fact_spending

ORDER BY total_spend DESC
Limit 1;
 than sol:
0	Meat Products	13537302.5



0	Meat Products	13537302.5
1	Wines	12419512
2	Fish Products	2526129
3	Gold Products	1710050
4	Sweet Products	833066
5	Fruits	511644




Q5:What is the relationship between income and total spending?

SELECT 
    d.customer_id,
    d.income,
    f.total_spend
FROM dim_customer d
JOIN fact_spending f
ON d.customer_id = f.customer_id
WHERE d.income IS NOT NULL
ORDER BY d.income DESC;

0	15094458	174948.58749999997	1396
1	15118576	174948.58749999997	1440
2	16489272	174948.58749999997	1281
3	9169447	174948.58749999997	1157
4	489682	174948.58749999997	215


CAMPAIGN PERFORMANCE INSIGHTS

Q What is the overall campaign response rate?

select 
   Round(avg(response)*100,2) As Response_rate_percentage
   from fact_campaign;


   Response_rate_percentage: 	14.76

select 
Count(CASE WHEN Response = 1 THEN 1 END) * 100.0 /count(*) As Response_rate_percentage
from fact_campaign;

Response_rate_percentage:	14.75893

Q2:Which campaign (CMP1–CMP5) performed best?


SELECT 'CMP1' AS campaign, SUM(accepted_cmp1) AS total_responses
FROM fact_campaign

UNION ALL

SELECT 'CMP2', SUM(accepted_cmp2)
FROM fact_campaign

UNION ALL

SELECT 'CMP3', SUM(accepted_cmp3)
FROM fact_campaign

UNION ALL

SELECT 'CMP4', SUM(accepted_cmp4)
FROM fact_campaign

UNION ALL

SELECT 'CMP5', SUM(accepted_cmp5)
FROM fact_campaign

ORDER BY total_responses DESC
Limit 1;



 total_responses: CMP1	7529



0	CMP1	7529
1	CMP3	3494
2	CMP4	3182
3	CMP5	2557
4	CMP2	807

Campaign 1 performed best

Q 2:Which country has the highest campaign response rate?

select  d.country,
           Round(avg(f.response)*100,2) As Response_rate_percent
    from dim_customer d
   join  fact_campaign f
on d.customer_id=f.customer_id
group by  d.country
order by Response_rate_percent;

0	Canada	11.57
1	Spain	12.93
2	Saudi Arabia	14.9
3	Mexico	15.2
4	USA	15.7
5	Germany	16.81
6	India	18.76
7	Australia	20.19

Output: Australia → 20.19% Highest Campaign Response rate



Q 3:Do high-income customers respond more to campaigns?


SELECT 
    customer_segment,
    response_segment,
    COUNT(*) AS total_customers
FROM fact_campaign
GROUP BY customer_segment, response_segment
ORDER BY customer_segment;

0	High Income	Non-Responder	8321
1	High Income	Responder	2196
2	High Spender	Non-Responder	10962
3	High Spender	Responder	3421
4	Normal	Non-Responder	28452
5	Normal	Responder	2648



SELECT 
    customer_segment,
    count(case when response_segment="Responder" Then 1 END) *100.0 as "Response"
 
FROM fact_campaign
GROUP BY customer_segment, response_segment
ORDER BY customer_segment;

0	High Income	      0
1	High Income	      219600
2	High Spender	  0
3	High Spender	  342100
4	Normal	          0
5	Normal	         264800